In [19]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
print("--- Loading Datasets ---")
# Using the popular TMDB 5000 dataset
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

--- Loading Datasets ---


In [21]:
# Merge the dataframes on the common identifier 'title'
movies = movies.merge(credits, on='title')

In [22]:
# Select features relevant to content-based filtering
# Budget, popularity, etc., are omitted to focus strictly on metadata content
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

In [23]:
# Handle missing values by dropping rows with null 'overview' strings
movies.dropna(inplace=True)
movies.reset_index(drop=True, inplace=True)

In [24]:
# STEP 2: Data Cleaning and Feature Parsing
# ---------------------------------------------------------------------
# The TMDB dataset stores genres, keywords, cast, and crew as stringified JSON lists.
# We need to parse them to extract actual names.

def convert_genres_keywords(obj):
    """Extracts names from the stringified JSON list of dictionaries."""
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

def convert_cast(obj):
    """Extracts the top 3 actors from the cast list."""
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

def fetch_director(obj):
    """Extracts the director's name from the crew list."""
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

print("--- Parsing JSON Metadata Features ---")
movies['genres'] = movies['genres'].apply(convert_genres_keywords)
movies['keywords'] = movies['keywords'].apply(convert_genres_keywords)
movies['cast'] = movies['cast'].apply(convert_cast)
movies['crew'] = movies['crew'].apply(fetch_director)


--- Parsing JSON Metadata Features ---


In [25]:
# To prevent vectorizer confusion (e.g., distinguishing "Johnny Depp" from "Johnny Knoxville"),
# we collapse spaces in names and multi-word genres/keywords.
def collapse_spaces(L):
    return [i.replace(" ", "") for i in L]

movies['genres'] = movies['genres'].apply(collapse_spaces)
movies['keywords'] = movies['keywords'].apply(collapse_spaces)
movies['cast'] = movies['cast'].apply(collapse_spaces)
movies['crew'] = movies['crew'].apply(collapse_spaces)

# Split the overview paragraph into a list of words
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [26]:
#STEP 3: Combine Text Features into a Single Metadata Field
# ---------------------------------------------------------------------
print("--- Creating Unified Metadata Tags ---")
# Create a single 'tags' column containing all structural and contextual text
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# Convert the list back into a single lowercase string for vectorization
cleaned_df = movies[['movie_id', 'title', 'tags']].copy()
cleaned_df['tags'] = cleaned_df['tags'].apply(lambda x: " ".join(x).lower())

--- Creating Unified Metadata Tags ---


In [27]:
# ---------------------------------------------------------------------
# STEP 4: Feature Extraction (Vectorization)
# ---------------------------------------------------------------------
print("--- Vectorizing Metadata Tags ---")
# CountVectorizer is preferred over TF-IDF here because recurring actor/director names
# or specific genres shouldn't be penalized by high document frequency.
cv = CountVectorizer(max_features=5000, stop_words='english')
vectorized_matrix = cv.fit_transform(cleaned_df['tags']).toarray()

--- Vectorizing Metadata Tags ---


In [16]:
# ---------------------------------------------------------------------
# STEP 5: Compute Cosine Similarity
# ---------------------------------------------------------------------
print("--- Computing Cosine Similarity Matrix ---")
# Computes pairwise cosine similarity between all vectors
similarity_matrix = cosine_similarity(vectorized_matrix)

--- Computing Cosine Similarity Matrix ---


In [28]:
# STEP 6: Implement Recommendation Function
# ---------------------------------------------------------------------
def recommend_movies(movie_title, top_n=10):
    """
    Finds and prints the top N movies similar to the given movie title.

    Parameters:
    movie_title (str): Precise title of the movie in the dataset.
    top_n (int): Number of recommendations to return.
    """
    # Check if movie exists in dataset
    if movie_title not in cleaned_df['title'].values:
        print(f"Error: '{movie_title}' not found in the dataset. Please check spelling.")
        return

    # Get the index of the input movie
    movie_index = cleaned_df[cleaned_df['title'] == movie_title].index[0]

    # Extract similarity scores for this movie and sort them in descending order
    # enumerate() keeps track of the original movie index
    distances = similarity_matrix[movie_index]
    similar_movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:top_n+1]

    # Display Results
    print(f"\n==================================================")
    print(f"TOP {top_n} RECOMMENDATIONS FOR: '{movie_title.upper()}'")
    print(f"==================================================")
    print(f"{'No.':<5}{'Movie Title':<35}{'Similarity Score':<15}")
    print(f"--------------------------------------------------")

    for rank, (idx, score) in enumerate(similar_movies_list, 1):
        title = cleaned_df.iloc[idx]['title']
        print(f"{rank:<5}{title:<35}{score:.4f}")

In [30]:
# STEP 7: Sample System Execution
# ---------------------------------------------------------------------
# Test the recommendation engine
recommend_movies('Batman Begins', top_n=5)


TOP 5 RECOMMENDATIONS FOR: 'BATMAN BEGINS'
No.  Movie Title                        Similarity Score
--------------------------------------------------
1    The Dark Knight                    0.3980
2    The Dark Knight Rises              0.3610
3    Batman                             0.3433
4    Batman                             0.3196
5    Batman & Robin                     0.3151
